In [0]:
%run ../connection

In [0]:
# =============================================================================
# NOTEBOOK: 02_bronze_autoloader.py
# LAYER:    Bronze (Structured Delta Tables)
# PURPOSE:  Use Databricks Autoloader (cloudFiles) to incrementally read
#           new JSON files from the ADLS Landing container and write them
#           into Bronze Delta tables — one table per endpoint.
#
# WHY AUTOLOADER (instead of spark.read)?
#   When you use spark.read.json(path), Spark reads ALL files at that path
#   every single time — even files from yesterday, last week, last month.
#   For a daily pipeline accumulating thousands of files, this is wasteful.
#
#   Autoloader solves this with a CHECKPOINT:
#     - On first run  → processes ALL files in the landing folder
#     - On subsequent runs → processes ONLY files added since the last run
#   This is called "incremental ingestion" and it scales infinitely.
#
# WHY NO HIVE METASTORE / CATALOG?
#   This project runs on a Databricks student account where:
#     - Public DBFS root is disabled  → CREATE DATABASE fails
#     - Unity Catalog is not provisioned → 3-part names (catalog.db.table) fail
#   Solution: access every Delta table directly by its ADLS abfss:// path.
#   spark.read.format("delta").load(PATH) works on ALL Databricks tiers
#   with zero metastore dependency. Silver notebooks follow the same pattern.
#
# HOW TO READ A BRONZE TABLE FROM ANOTHER NOTEBOOK (Silver etc.):
#   df = spark.read.format("delta").load(
#       f"abfss://bronze@<account>.dfs.core.windows.net/coins_markets_raw"
#   )
#   No database name needed — just the path.
#
# WHAT THIS NOTEBOOK PRODUCES:
#   Five Bronze Delta table folders in the ADLS bronze/ container:
#     bronze/coins_markets_raw/   → one row per daily run file
#     bronze/ohlc_raw/            → one row per coin OHLC file
#     bronze/market_chart_raw/    → one row per coin market_chart file
#     bronze/trending_raw/        → one row per trending snapshot file
#     bronze/global_raw/          → one row per global sn

In [0]:
%run ./config

In [0]:
# =============================================================================
# CELL 3 — AUTOLOADER HELPER FUNCTION
#
# THE FULL AUTOLOADER FLOW PER ENDPOINT:
#
#   spark.readStream(cloudFiles)         ← detects & reads ONLY new JSON files
#           │
#   .withColumn(bronze_loaded_at)        ← when did Bronze receive this row?
#   .withColumn(source_file_path)        ← which Landing file did it come from?
#   .withColumn(source_file_name)        ← just the filename (no full path)
#           │
#   .writeStream(delta, append)          ← appends to Bronze Delta on ADLS
#           │
#   checkpoint updated                   ← "I have seen these files" state saved
#
# KEY SETTINGS EXPLAINED:
#
#   trigger(availableNow=True)
#     Process all currently pending files then STOP the stream.
#     This turns the streaming API into a clean batch operation.
#     Without this, writeStream runs forever — wrong for daily pipelines.
#     availableNow=True is the modern replacement for the deprecated
#     trigger(once=True).
#
#   outputMode("append")
#     Bronze is append-only. Each run adds rows, nothing is ever updated.
#     Deduplication and upserts happen in Silver via Delta MERGE.
#
#   recursiveFileLookup=True
#     Needed for ohlc and market_chart where files sit inside
#     coin_id/YYYY/MM/DD/ sub-folders. Without this, Autoloader only
#     looks at the top-level of the landing path and finds nothing.
#
#   cloudFiles.schemaLocation (unique per endpoint)
#     Autoloader writes its inferred schema here on first run.
#     Must be a unique path per endpoint — sharing would cause schema conflicts.
# =============================================================================
 
def run_autoloader(
    endpoint_name     : str,
    landing_subfolder : str,
    bronze_table_path : str,
) -> dict:
    """
    Run Databricks Autoloader for one CoinGecko endpoint.
 
    Reads new JSON files from LANDING_ROOT/landing_subfolder, enriches them
    with Bronze audit columns, and appends to the Delta table at
    bronze_table_path (accessed by path — no catalog registration).
 
    Returns a dict with row_count and columns for the validation cell.
 
    Args:
        endpoint_name     : Short label for checkpoint/schema sub-paths and logs
                            (e.g., "coins_markets", "ohlc")
        landing_subfolder : Sub-path under LANDING_ROOT to read from
                            (e.g., "coins_markets", "ohlc", "trending")
        bronze_table_path : Full abfss:// path for the Bronze Delta table output
    """
    checkpoint_path = f"{CHECKPOINT_ROOT}/{endpoint_name}"
    schema_path     = f"{SCHEMA_ROOT}/{endpoint_name}"
    landing_path    = f"{LANDING_ROOT}/{landing_subfolder}"
 
    logger.info(f"  ┌─ Autoloader: {endpoint_name}")
    logger.info(f"  │  Landing  : {landing_path}")
    logger.info(f"  │  Bronze   : {bronze_table_path}")
    logger.info(f"  │  Checkpoint: {checkpoint_path}")
 
    # ── READ STREAM ───────────────────────────────────────────────────────────
    raw_stream = (
        spark.readStream
             .format("cloudFiles")
             .options(**{
                 # Source file format
                 "cloudFiles.format"           : "json",
 
                 # Schema cache location — unique per endpoint (not shared)
                 "cloudFiles.schemaLocation"   : schema_path,
 
                 # Infer actual data types (int, double, boolean) from JSON values.
                 # Without this, every column becomes StringType.
                 "cloudFiles.inferColumnTypes" : "true",
 
                 # Our JSON files are formatted with indent=2 (pretty-printed).
                 # multiLine=true tells Spark the whole file is one JSON document,
                 # not one JSON object per line (JSONL format).
                 "multiLine"                   : "true",
 
                 # Walk sub-directories automatically.
                 # Essential for ohlc/{coin_id}/YYYY/MM/DD/ folder structure.
                 "recursiveFileLookup"         : "true",
 
                 # Only pick up .json files — ignores any Autoloader internal
                 # temp files or checkpoint files that might appear in same paths.
                 "pathGlobFilter"              : "*.json",
             })
             .load(landing_path)
    )
 
    # ── ENRICH WITH BRONZE AUDIT COLUMNS ─────────────────────────────────────
    # These columns are added BY OUR PIPELINE, not from the API.
    # They answer three lineage questions:
    #   1. When did this row arrive in Bronze?  → bronze_loaded_at
    #   2. Which Landing file produced it?      → source_file_path
    #   3. What is just the filename?           → source_file_name
    #
    # _metadata.file_path is a hidden Spark metadata column available
    # when reading files via readStream or read. It gives the full ADLS
    # path of the source file for every row — perfect for data lineage.
    enriched_stream = (
        raw_stream
        .withColumn(
            "bronze_loaded_at",
            # F.lit() creates a constant value column.
            # We fix it to NOW_UTC so all rows in this run share the same
            # timestamp — easier to filter "all rows loaded in run X" later.
            # current_timestamp() would vary slightly per micro-batch.
            F.lit(NOW_UTC.isoformat()).cast(TimestampType())
        )
        .withColumn(
            "source_file_path",
            F.col("_metadata.file_path")     # Full abfss:// path of the source file
        )
        .withColumn(
            "source_file_name",
            # Split path by "/" and take the last element = filename only
            # e.g., "coins_markets_20241115_060000.json"
            F.element_at(F.split(F.col("_metadata.file_path"), "/"), -1)
        )
    )
 
    # ── WRITE STREAM → BRONZE DELTA (path-based, NO catalog) ─────────────────
    query = (
        enriched_stream
        .writeStream
        .format("delta")
        .outputMode("append")                       # Append-only: Bronze never updates rows
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")              # Auto-add new columns from API changes
        .trigger(availableNow=True)                 # Process pending files then stop
        .start(bronze_table_path)                   # Write to ADLS path — no catalog name
    )
 
    # CRITICAL: block until the streaming query fully finishes.
    # Without this, Python moves to the next cell immediately while
    # Spark is still writing — validation would see 0 rows.
    query.awaitTermination()
 
    # ── READ BACK TO GET STATS (path-based) ──────────────────────────────────
    df    = spark.read.format("delta").load(bronze_table_path)
    count = df.count()
    cols  = df.columns
 
    logger.info(f"  └─ ✓ Done | rows: {count:,} | columns: {len(cols)}")
    return {"path": bronze_table_path, "row_count": count, "columns": cols}
 
 

In [0]:
# =============================================================================
# CELL 4 — OPTIMIZE HELPER
#
# WHY OPTIMIZE + Z-ORDER?
#   Each daily Autoloader run appends a new batch of small Parquet files.
#   After 30 days you have 30 small files instead of 1 large file.
#   Small files are slow: each requires a separate ADLS API call to open.
#
#   OPTIMIZE merges small files into fewer, larger Parquet files (faster reads).
#
#   Z-ORDER BY <column> physically clusters rows by that column value on disk.
#   Example: Z-ORDER BY coin_id means all "bitcoin" rows are in adjacent files.
#   A Silver query WHERE coin_id = 'bitcoin' then skips 90%+ of files entirely.
#   This is called "data skipping" — Delta's core performance feature.
#
# SYNTAX NOTE — delta.`path`:
#   When using spark.sql() with a full ADLS path (instead of a catalog table),
#   you MUST prefix with "delta." and wrap the path in backticks.
#   This tells Spark "this is a Delta table at this path" without any catalog.
#   Example: OPTIMIZE delta.`abfss://bronze@account.dfs.core.windows.net/ohlc_raw`
# =============================================================================
 
def optimize_delta(bronze_table_path: str, zorder_col: str, label: str) -> None:
    """
    Run OPTIMIZE + Z-ORDER on a Bronze Delta table accessed by ADLS path.
    Uses the delta.`path` syntax — no catalog or metastore required.
 
    Args:
        bronze_table_path : Full abfss:// Delta table path
        zorder_col        : Column to Z-ORDER by (most common filter column in Silver)
        label             : Short name for log messages
    """
    logger.info(f"  Optimizing {label} (Z-ORDER BY {zorder_col}) ...")
 
    # delta.`<path>` syntax: works on all Databricks tiers, no metastore needed.
    # Backticks are required because the path contains @, /, and . characters.
    spark.sql(f"OPTIMIZE delta.`{bronze_table_path}` ZORDER BY ({zorder_col})")
 
    logger.info(f"  ✓ {label} optimized")
 
 

In [0]:
# =============================================================================
# CELL 5 — RUN AUTOLOADER FOR ALL 5 ENDPOINTS
# =============================================================================
 
results = {}
 
# ── 1/5: coins/markets ───────────────────────────────────────────────────────
# Landing : landing/coins_markets/YYYY/MM/DD/coins_markets_*.json
# Payload : list of 50 coin market dicts
logger.info("=" * 70)
logger.info("AUTOLOADER 1/5: coins_markets")
results["coins_markets_raw"] = run_autoloader(
    endpoint_name     = "coins_markets",
    landing_subfolder = "coins_markets",
    bronze_table_path = f"{BRONZE_ROOT}/coins_markets_raw",
)
 
# ── 2/5: ohlc ────────────────────────────────────────────────────────────────
# Landing : landing/ohlc/{coin_id}/YYYY/MM/DD/ohlc_{coin_id}_*.json
# Payload : list of [timestamp_ms, open, high, low, close] arrays
# coin_id is in the envelope extra_meta field (added by notebook 01)
logger.info("=" * 70)
logger.info("AUTOLOADER 2/5: ohlc (all coins, recursive subfolders)")
results["ohlc_raw"] = run_autoloader(
    endpoint_name     = "ohlc",
    landing_subfolder = "ohlc",
    bronze_table_path = f"{BRONZE_ROOT}/ohlc_raw",
)
 
# ── 3/5: market_chart ────────────────────────────────────────────────────────
# Landing : landing/market_chart/{coin_id}/YYYY/MM/DD/market_chart_{coin_id}_*.json
# Payload : {prices: [[ts,val],...], market_caps: [...], total_volumes: [...]}
logger.info("=" * 70)
logger.info("AUTOLOADER 3/5: market_chart (all coins, recursive subfolders)")
results["market_chart_raw"] = run_autoloader(
    endpoint_name     = "market_chart",
    landing_subfolder = "market_chart",
    bronze_table_path = f"{BRONZE_ROOT}/market_chart_raw",
)
 
# ── 4/5: trending ────────────────────────────────────────────────────────────
# Landing : landing/trending/YYYY/MM/DD/trending_*.json
# Payload : {coins: [{item: {id, name, symbol, score, ...}}, ...]}
logger.info("=" * 70)
logger.info("AUTOLOADER 4/5: trending")
results["trending_raw"] = run_autoloader(
    endpoint_name     = "trending",
    landing_subfolder = "trending",
    bronze_table_path = f"{BRONZE_ROOT}/trending_raw",
)
 
# ── 5/5: global ──────────────────────────────────────────────────────────────
# Landing : landing/global/YYYY/MM/DD/global_*.json
# Payload : {data: {total_market_cap: {usd:...}, market_cap_percentage: {btc:...}, ...}}
logger.info("=" * 70)
logger.info("AUTOLOADER 5/5: global")
results["global_raw"] = run_autoloader(
    endpoint_name     = "global",
    landing_subfolder = "global",
    bronze_table_path = f"{BRONZE_ROOT}/global_raw",
)
 
 

In [0]:
# =============================================================================
# CELL 6 — VALIDATION
#
# Every Bronze table must have:
#   a) At least 1 row  (confirms data was actually written by Autoloader)
#   b) Envelope columns from Landing (proves lineage chain is intact)
#   c) Bronze audit columns we added in this notebook
#
# Failure → raises ValueError → notebook exits non-zero →
# ADF marks the activity as FAILED → failure alert fires.
# This prevents silent failures where an empty Bronze table
# flows into Silver and produces equally empty Silver tables.
# =============================================================================
 
logger.info("=" * 70)
logger.info("VALIDATION: Checking all 5 Bronze Delta tables")
 
# Columns that must exist from the Landing envelope (notebook 01)
REQUIRED_ENVELOPE_COLS = {"pipeline_run_id", "ingestion_timestamp", "source_endpoint"}
# Columns that we add in this notebook
REQUIRED_BRONZE_COLS   = {"bronze_loaded_at", "source_file_path", "source_file_name"}
 
validation_summary = {}
failed_tables      = []
 
for table_name, result in results.items():
    col_set          = set(result["columns"])
    row_count        = result["row_count"]
    missing_envelope = REQUIRED_ENVELOPE_COLS - col_set
    missing_bronze   = REQUIRED_BRONZE_COLS   - col_set
    has_data         = row_count > 0
    passed           = has_data and not missing_envelope and not missing_bronze
    status           = "PASS" if passed else "FAIL"
 
    validation_summary[table_name] = {
        "status"      : status,
        "row_count"   : row_count,
        "missing_cols": list(missing_envelope | missing_bronze),
    }
 
    logger.info(
        f"  [{status}] {table_name:<25} "
        f"rows: {row_count:>6,}  "
        f"missing: {list(missing_envelope | missing_bronze) or 'none'}"
    )
 
    if not passed:
        failed_tables.append(table_name)
 
if failed_tables:
    raise ValueError(
        f"Bronze validation FAILED for: {failed_tables}\n"
        f"Details:\n{json.dumps(validation_summary, indent=2)}"
    )
 
logger.info("  ✓ All 5 Bronze tables passed validation")
 
 

In [0]:
# =============================================================================
# CELL 7 — OPTIMIZE + Z-ORDER
#
# Run after validation so we only compact tables confirmed to have good data.
# Z-ORDER column choices rationale:
#   coins_markets_raw → pipeline_run_id  (Silver reads latest run's snapshot)
#   ohlc_raw          → coin_id          (Silver explodes and filters per coin)
#   market_chart_raw  → coin_id          (Silver explodes and filters per coin)
#   trending_raw      → pipeline_run_id  (Silver reads latest run's trending list)
#   global_raw        → pipeline_run_id  (Silver reads latest run's global stats)
# =============================================================================
 
logger.info("=" * 70)
logger.info("OPTIMIZE: Compacting Bronze Delta tables")
 
optimize_delta(f"{BRONZE_ROOT}/coins_markets_raw", "pipeline_run_id", "coins_markets_raw")
optimize_delta(f"{BRONZE_ROOT}/ohlc_raw",           "coin_id",         "ohlc_raw")
optimize_delta(f"{BRONZE_ROOT}/market_chart_raw",   "coin_id",         "market_chart_raw")
optimize_delta(f"{BRONZE_ROOT}/trending_raw",       "pipeline_run_id", "trending_raw")
optimize_delta(f"{BRONZE_ROOT}/global_raw",         "pipeline_run_id", "global_raw")
 
logger.info("  ✓ All tables optimized")
 

In [0]:
# =============================================================================
# CELL 8 — COMPLETION SUMMARY
# =============================================================================
 
summary = {
    "event"            : "bronze_autoloader_complete",
    "run_timestamp_utc": NOW_UTC.isoformat(),
    "tables_loaded"    : len(results),
    "row_counts"       : {t: r["row_count"] for t, r in results.items()},
    "validation"       : {t: v["status"]    for t, v in validation_summary.items()},
    "status"           : "SUCCESS",
}
 
logger.info("=" * 70)
logger.info("BRONZE AUTOLOADER COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<28}: {v}")
logger.info("=" * 70)
logger.info("")
logger.info("HOW TO READ THESE TABLES IN SILVER NOTEBOOK:")
logger.info("  # No catalog or database needed — just the ADLS path")
logger.info("  df = spark.read.format('delta').load(")
logger.info(f"      'abfss://bronze@{ADLS_NAME}.dfs.core.windows.net/coins_markets_raw'")
logger.info("  )")
 